# Lesson 6 : Memory and personalization (Context Provider)

By using memory, you can keep the context in agent's conversation. For example, the assistant may keep your food's preference from past conversations, and it might recommend you the restaurant you favor when planning a trip later.

Context provider in Agent Framework handles context by using memory (for both short-term ephemeral memory and long-term persisted memory).  
In Agent Framework, you can use built-in context providers - Azure AI search, Mem0, or Redis as backend -, or you can configure your own custom provider.

In this exercise, we explore context provider with Redis backend.

It's worth noting that **currently RediSearch doesn't support Japanese full text search**. Do not use Japanese prompt and instruction in this exercise. (Use other provider for Japanese language.)

> Note : Here I don't go so far, but you can also configure Mem0 with Azure AI search or Redis backend as a vector store. (Today Mem0 is very popular for implementing memory.)

> Note : When you're using ```AzureAIClient```, you can also use memory search tool (type : ```memory_search```) in Microsoft Foundry for handling memory. (See [here](https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models.memorysearchtool?view=azure-python-preview) for API reference.)  
> See [Lesson 5](./05_foundry_tools.ipynb) for using Foundry tools.

## 1. Prepare Redis cache

Before writing code, we should provision Redis - by OSS installation or managed service.  
In this example, we use Azure Managed Redis, which is based on Redis Enterprise.

Open [Azure Portal](https://portal.azure.com) and create a new "Azure Managed Redis" resource with the following settings.

- Set "No Eviction" as "Eviction Policy".
- Select "Enterprise" as "Clustering Policy".
- Select "RediSearch" in "Modules".
- Select "Enable public access from all networks" in "Network access".
- Enable "Access Keys Authentication".

After Redis instance is created, copy endpoint url and access key in the created resource on Azure Portal.

## 2. Run agent with context provider

Now let's build an agent to use Redis context provider.

First we initilize the client object as usual.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we create an agent with Redis context provider as follows.  
Before running code, set Redis endpoint (which is retreived above) in ```REDIS_HOST```, and set access key (which is also retreived above) in ```REDIS_PASSWORD```.

> Note : In this example, we write settings directly into the code for demo purpose. (For security reasons, set these settings in environment variable and use it in production.)

**In this example, I have simply configured text-only retrieval for demo purpose**. (This example won't then perform semantic retrieval.)  
In production, please configure hybrid (text + vector) retrieval.

In [2]:
from agent_framework import Agent
from agent_framework.redis import RedisContextProvider

# ToDo : fill your settings
REDIS_HOST = "xxxxx.xxxxx.redis.azure.net:10000"
REDIS_PASSWORD = "xxxxxxxxxxxxxxxxxxxx"

redis_url = f"rediss://:{REDIS_PASSWORD}@{REDIS_HOST}"

provider1 = RedisContextProvider(
    redis_url=redis_url,
    source_id="redis_basic_chat",
    index_name="index01",
    application_id="app01",
    agent_id="agent01",
    user_id="demouser01",
)

agent1 = Agent(
    name="AgentWithContext01",
    client=client,
    instructions=(
        "You are a helpful assistant."
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider1],
)

Now let's run agent with the following user message which tells your food preference.  
In the backend, Redis index is generated and the message context is stored in the index.

In [3]:
from IPython.display import Markdown, display

result = await agent1.run("My favorite food is sushi.")
display(Markdown(result.text))

I don’t have any stored context from earlier in this chat.

Noted: your favorite food is sushi.

Next let's ask my favor in this agent as follows.  
In the backend, the agent retrieves the context associated with this message, and this context is used to create a response. (As I have mentioned above, text-only retrieval is applied in this example. So direct question (primitive question) is issued in this example.)

In [4]:
result = await agent1.run("What is my favorite food?")
display(Markdown(result.text))

Your favorite food is sushi.

You can scope the context by using ```application_id```, ```agent_id```, ```user_id``` and ```thread_id``` in ```RedisProvider``` configuration.

Now I create a new agent which has context provider with different user id "demouser02".  
As you see, the agent doesn't know my preference, because it's a context of different user.

In [5]:
provider2 = RedisContextProvider(
    redis_url=redis_url,
    source_id="redis_basic_chat",
    index_name="index01",
    application_id="app01",
    agent_id="agent01",
    user_id="demouser02",
)

agent2 = Agent(
    name="AgentWithContext02",
    client=client,
    instructions=(
        "You are a helpful assistant."
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider2],
)

result = await agent2.run("What is my favorite food?")
display(Markdown(result.text))

I don’t have any stored context about your favorite food, so I can’t know it yet. Tell me what it is (or give me a few candidates), and I’ll remember it for this chat.

Now we go back again to the user "demouser01", and ask the preference.  
The agent object is newly created (therefore, started a new thread), but it remembers my preference, because it's stored in the context provider.

In [6]:
client = AzureAIClient(
    credential=credential,
    agent_name="AgentWithContext01",
    agent_version="1",
)

agent1 = Agent(
    client=client,
    instructions=(
        "You are a helpful assistant."
        "Before answering, always check for stored context."
    ),
    tools=[],
    context_providers=[provider1],
)

result = await agent1.run("What is my favorite food?")
display(Markdown(result.text))

Your favorite food is sushi.

If you want to see the stored messages in Redis index for debugging, you can run as follows.

In [7]:
all_data = await provider1.search_all()
for item in all_data:
    print(item)

{'id': 'context:01KF008VBT6TDZMS1GFH9GKRTR', 'role': 'user', 'content': 'My favorite food is sushi.', 'conversation_id': 'resp_0c2d274c219e471f00696873d0807081909b0b018ad6f1e65c', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01'}
{'id': 'context:01KF008VBX93Y1GCEV1Y6JT7MP', 'role': 'assistant', 'content': 'I don’t have any stored context from earlier in this chat.\n\nNoted: your favorite food is sushi.', 'conversation_id': 'resp_0c2d274c219e471f00696873d0807081909b0b018ad6f1e65c', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01', 'author_name': 'AgentWithContext01'}
{'id': 'context:01KF0097JSCTE5SR75V61CRSN4', 'role': 'user', 'content': 'What is my favorite food?', 'conversation_id': 'resp_08441b9242cb055f00696873ddac348197a04573ea30aa9e58', 'application_id': 'app01', 'agent_id': 'agent01', 'user_id': 'demouser01'}
{'id': 'context:01KF0097JSCTE5SR75V61CRSN5', 'role': 'assistant', 'content': 'Your favorite food is sushi.', 'conversation_

If required, you can explicitly remove the index as follows.

In [8]:
await provider1.redis_index.delete()